In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from dataclasses import dataclass

@dataclass
class Context:
    user_name: str
    age: int

In [4]:
from langchain.agents.middleware import before_model

@before_model
def log_before_model(state, runtime):
    print("-" * 30)
    print("state:", state)
    print("runtime:", runtime)
    print("-" * 30)
    return None

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    middleware=[log_before_model],
    context_schema=Context
)

In [6]:
agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐야?"}]},
    context=Context(user_name="김일남", age=99)
)

------------------------------
state: {'messages': [HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='204f64b6-aab6-4a5e-a1f6-b50a3e8f9303')]}
runtime: Runtime(context=Context(user_name='김일남', age=99), store=None, stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x0000016694BED440>, heartbeat=<function _no_op_heartbeat at 0x000001669338F100>, previous=None, execution_info=ExecutionInfo(checkpoint_id='1f15cb45-f389-6080-8000-fe1ba9fc74c4', checkpoint_ns='log_before_model.before_model:fda37a3f-4276-1472-e9b6-136f19d2687d', task_id='fda37a3f-4276-1472-e9b6-136f19d2687d', thread_id=None, run_id=None, node_attempt=1, node_first_attempt_time=1780206518.6607232), server_info=None, control=<langgraph.runtime.RunControl object at 0x0000016695E78A90>)
------------------------------


{'messages': [HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='204f64b6-aab6-4a5e-a1f6-b50a3e8f9303'),
  AIMessage(content=[{'type': 'text', 'text': '죄송하지만, 저는 사용자의 개인정보를 저장하거나 기억할 수 없기 때문에 고객님의 성함을 알지 못합니다. \n\n혹시 이전에 말씀해 주셨더라도, 보안과 개인정보 보호를 위해 대화가 종료되면 관련 정보를 유지하지 않으니 양해 부탁드립니다. 저에게 이름을 알려주시면 대화하는 동안 불러드릴게요!', 'extras': {'signature': 'EjQKMgEMOdbH6Av8thVrmgSfqh0B2slYTYM6A6Br/kU4FOT68meqC2Exxg6dpE725ZxLXMJe'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7c93-c188-7433-8bbb-4a20d23ec4dd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 76, 'total_tokens': 82, 'input_token_details': {'cache_read': 0}})]}

In [8]:
from langchain.agents.middleware import wrap_model_call

@wrap_model_call
def inject_user_name(request, handler):
    print(f"request:", request)
    print("-" * 10)

    user_name = request.runtime.context.user_name

    if user_name:
        sys_prompt = f"사용자의 이름은 [{user_name}]입니다."
    else:
        sys_prompt = "사용자의 이름은 알려지지 않았습니다."
    
    request = request.override(system_prompt=sys_prompt)

    return handler(request)

In [11]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    middleware=[log_before_model, inject_user_name],
    context_schema=Context
)

In [12]:
agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐야?"}]},
    context=Context(user_name="김일남", age=99)
)

------------------------------
state: {'messages': [HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='75649e48-f6d6-4e1a-81b8-b4c8fc5081ff')]}
runtime: Runtime(context=Context(user_name='김일남', age=99), store=None, stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x0000016694B6A160>, heartbeat=<function _no_op_heartbeat at 0x000001669338F100>, previous=None, execution_info=ExecutionInfo(checkpoint_id='1f15cb74-bd74-6c97-8000-ab7040bb6e7d', checkpoint_ns='log_before_model.before_model:77f9efac-6c83-70f0-5b14-57e3d3c29c1f', task_id='77f9efac-6c83-70f0-5b14-57e3d3c29c1f', thread_id=None, run_id=None, node_attempt=1, node_first_attempt_time=1780207774.6367638), server_info=None, control=<langgraph.runtime.RunControl object at 0x0000016695E6A8C0>)
------------------------------
request: ModelRequest(model=ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 3.1 Flash Lite', 'release_date': '2026-05-07', 'last_updated': '2026-05-0

{'messages': [HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='75649e48-f6d6-4e1a-81b8-b4c8fc5081ff'),
  AIMessage(content=[{'type': 'text', 'text': '당신의 이름은 **김일남** 님입니다. 무엇을 도와드릴까요?', 'extras': {'signature': 'EjQKMgEMOdbHS2k9vtbduUWUa19BAF5fI+IXwLOY8U5PVAhxhDgKOVwcAJYdvWY3IlX8PjED'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7ca6-ebb0-7190-bc9e-699fa0805955-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 19, 'total_tokens': 36, 'input_token_details': {'cache_read': 0}})]}

In [13]:
from langchain.agents.middleware import after_model

@after_model
def log_after_model(state, runtime):
    print(f"after_model_state: {state}")
    print("-" * 10)
    return None

In [14]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    middleware=[log_before_model, inject_user_name, log_after_model],
    context_schema=Context
)

agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐야?"}]},
    context=Context(user_name="김일남", age=99)
)

------------------------------
state: {'messages': [HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='d49e196f-ef97-4898-92df-5fa531887a06')]}
runtime: Runtime(context=Context(user_name='김일남', age=99), store=None, stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x0000016694CE2480>, heartbeat=<function _no_op_heartbeat at 0x000001669338F100>, previous=None, execution_info=ExecutionInfo(checkpoint_id='1f15cb7d-8bf6-65e7-8000-306c33c6843e', checkpoint_ns='log_before_model.before_model:b03466f1-b78e-58a9-b890-28105c9ed330', task_id='b03466f1-b78e-58a9-b890-28105c9ed330', thread_id=None, run_id=None, node_attempt=1, node_first_attempt_time=1780208011.038871), server_info=None, control=<langgraph.runtime.RunControl object at 0x0000016694967C40>)
------------------------------
request: ModelRequest(model=ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 3.1 Flash Lite', 'release_date': '2026-05-07', 'last_updated': '2026-05-07

{'messages': [HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='d49e196f-ef97-4898-92df-5fa531887a06'),
  AIMessage(content=[{'type': 'text', 'text': '사용자님의 성함은 [김일남] 님입니다.', 'extras': {'signature': 'EjQKMgEMOdbHGwvp8nzyOF1I/tTrow9SH/uf4KOYsElF4JmM57+z+wQFKCqeG4riok/ZHFuD'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7caa-8721-7cc0-985b-66a4676adfc3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 15, 'total_tokens': 32, 'input_token_details': {'cache_read': 0}})]}